## **1) Monstruos en las 10 habitaciones con más comentarios bug**

```python
[
  {
    $match: {
      hints: {
        $ne: null
      }
    }
  },
  {
    $addFields: {
      n_hints_bugs: {
        $size: {
          $filter: {
            input: "$hints",
            as: "hint",
            cond: {
              $eq: ["$$hint.category", "bug"]
            }
          }
        }
      }
    }
  },
  {
    $sort: {
      n_hints_bugs: -1
    }
  },
  {
    $limit: 10
  },
  {
    $unwind: "$monsters"
  },
  {
    $group: {
      _id: "$monsters.id",
      name: {
        $first: "$monsters.name"
      }
    }
  }
]

## **2) Mazmorras con comentarios hint por encima de la media**

```python
[
  {
    $addFields: {
      n_hints: {
        $size: {
          $filter: {
            input: "$hints",
            as: "hint",
            cond: {
              $eq: ["$$hint.category", "hint"]
            }
          }
        }
      }
    }
  },
  {
    $group: {
      _id: "$dungeon_id",
      dungeon_name: {
        $first: "$dungeon_name"
      },
      total_hints_mazmorra: {
        $sum: "$n_hints"
      }
    }
  },
  {
    $group: {
      _id: null,
      media_global: {
        $avg: "$total_hints_mazmorra"
      },
      mazmorras: {
        $push: {
          nombre: "$dungeon_name",
          total: "$total_hints_mazmorra"
        }
      }
    }
  },
  {
    $unwind: "$mazmorras"
  },
  {
    $match: {
      $expr: {
        $gt: ["$mazmorras.total", "$media_global"]
      }
    }
  },
  {
    $project: {
      _id: 0,
      dungeon_name: "$mazmorras.nombre"
    }
  }
]

## **3) Por mazmorra: comentarios por tipo, oro total, nivel mediano y usuarios por país**

```python
[
    {
        '$addFields': {
            'oro_habitacion': {
                '$sum': '$Loot.gold'
            }
        }
    }, {
        '$group': {
            '_id': '$dungeon_id', 
            'dungeon_name': {
                '$first': '$dungeon_name'
            }, 
            'oro_total': {
                '$sum': '$oro_habitacion'
            }, 
            'lista_categorias': {
                '$push': '$hints.category'
            }, 
            'lista_paises': {
                '$push': '$hints.publish_by.country'
            }, 
            'lista_niveles': {
                '$push': '$monsters.level'
            }
        }
    }, {
        '$project': {
            'dungeon_name': 1, 
            'oro_total': 1, 
            'categorias': {
                '$reduce': {
                    'input': '$lista_categorias', 
                    'initialValue': [], 
                    'in': {
                        '$concatArrays': [
                            '$$value', '$$this'
                        ]
                    }
                }
            }, 
            'paises': {
                '$reduce': {
                    'input': '$lista_paises', 
                    'initialValue': [], 
                    'in': {
                        '$concatArrays': [
                            '$$value', '$$this'
                        ]
                    }
                }
            }, 
            'niveles': {
                '$reduce': {
                    'input': '$lista_niveles', 
                    'initialValue': [], 
                    'in': {
                        '$concatArrays': [
                            '$$value', '$$this'
                        ]
                    }
                }
            }
        }
    }, {
        '$addFields': {
            'niveles_ordenados': {
                '$sortArray': {
                    'input': '$niveles', 
                    'sortBy': 1
                }
            }
        }
    }, {
        '$addFields': {
            'nivel_mediano': {
                '$arrayElemAt': [
                    '$niveles_ordenados', {
                        '$floor': {
                            '$divide': [
                                {
                                    '$size': '$niveles_ordenados'
                                }, 2
                            ]
                        }
                    }
                ]
            }
        }
    }, {
        '$unwind': {
            'path': '$categorias', 
            'preserveNullAndEmptyArrays': True
        }
    }, {
        '$group': {
            '_id': {
                'dungeon': '$_id', 
                'categoria': '$categorias'
            }, 
            'count_cat': {
                '$sum': {
                    '$cond': [
                        {
                            '$ifNull': [
                                '$categorias', False
                            ]
                        }, 1, 0
                    ]
                }
            }, 
            'dungeon_name': {
                '$first': '$dungeon_name'
            }, 
            'oro_total': {
                '$first': '$oro_total'
            }, 
            'paises': {
                '$first': '$paises'
            }, 
            'nivel_mediano': {
                '$first': '$nivel_mediano'
            }
        }
    }, {
        '$group': {
            '_id': '$_id.dungeon', 
            'comentarios_por_tipo': {
                '$push': {
                    '$cond': [
                        {
                            '$ifNull': [
                                '$_id.categoria', False
                            ]
                        }, {
                            'tipo': '$_id.categoria', 
                            'total': '$count_cat'
                        }, '$$REMOVE'
                    ]
                }
            }, 
            'dungeon_name': {
                '$first': '$dungeon_name'
            }, 
            'oro_total': {
                '$first': '$oro_total'
            }, 
            'paises': {
                '$first': '$paises'
            }, 
            'nivel_mediano': {
                '$first': '$nivel_mediano'
            }
        }
    }, {
        '$unwind': {
            'path': '$paises', 
            'preserveNullAndEmptyArrays': True
        }
    }, {
        '$group': {
            '_id': {
                'dungeon': '$_id', 
                'pais': '$paises'
            }, 
            'count_pais': {
                '$sum': {
                    '$cond': [
                        {
                            '$ifNull': [
                                '$paises', False
                            ]
                        }, 1, 0
                    ]
                }
            }, 
            'dungeon_name': {
                '$first': '$dungeon_name'
            }, 
            'oro_total': {
                '$first': '$oro_total'
            }, 
            'comentarios_por_tipo': {
                '$first': '$comentarios_por_tipo'
            }, 
            'nivel_mediano': {
                '$first': '$nivel_mediano'
            }
        }
    }, {
        '$group': {
            '_id': '$_id.dungeon', 
            'usuarios_por_pais': {
                '$push': {
                    '$cond': [
                        {
                            '$ifNull': [
                                '$_id.pais', False
                            ]
                        }, {
                            'pais': '$_id.pais', 
                            'total': '$count_pais'
                        }, '$$REMOVE'
                    ]
                }
            }, 
            'dungeon_name': {
                '$first': '$dungeon_name'
            }, 
            'oro_total': {
                '$first': '$oro_total'
            }, 
            'comentarios_por_tipo': {
                '$first': '$comentarios_por_tipo'
            }, 
            'nivel_mediano': {
                '$first': '$nivel_mediano'
            }
        }
    }, {
        '$project': {
            '_id': 0, 
            'dungeon_name': 1, 
            'oro_total': 1, 
            'nivel_mediano': 1, 
            'comentarios_por_tipo': 1, 
            'usuarios_por_pais': 1
        }
    }
]

## Query 4

```python
[
  {
    $unwind: "$in_rooms"
  },
  {
    $group: {
      _id: "$type",
      dungeons: {
        $addToSet: "$in_rooms.dungeon_name"
      }
    }
  },
  {
    $project: {
      _id: 0,
      monster_type: "$_id",
      dungeons: 1
    }
  }
]

## Query 5


```python
[
  {
    $unwind: "$in_rooms"
  },
  {
    $group: {
      _id: {
        dungeon: "$in_rooms.dungeon_name",
        type: "$type"
      },
      cantidad_total: {
        $sum: "$in_rooms.amount"
      }
    }
  },
  {
    $sort: {
      "_id.dungeon": 1,
      cantidad_total: -1
    }
  },
  {
    $group: {
      _id: "$_id.dungeon",
      tipo_mas_comun: {
        $first: "$_id.type"
      }
    }
  },
  {
    $project: {
      _id: 0,
      dungeon_name: "$_id",
      tipo_mas_comun: 1
    }
  }
]

## Query 6

```python 
[
  {
    $addFields: {
      nivel_medio_encuentro: {
        $avg: "$monsters.level"
      }
    }
  },
  {
    $group: {
      _id: null,
      maximo_absoluto: {
        $max: "$nivel_medio_encuentro"
      },
      habitaciones: {
        $push: {
          nivel_medio: "$nivel_medio_encuentro",
          comentarios: "$hints"
        }
      }
    }
  },
  {
    $unwind: "$habitaciones"
  },
  {
    $match: {
      $expr: {
        $eq: [
          "$habitaciones.nivel_medio",
          "$maximo_absoluto"
        ]
      }
    }
  },
  {
    $project: {
      _id: 0,
      comentarios: "$habitaciones.comentarios"
    }
  }
]

## Query 7

```python
[
  {
    $match: {
      loot: {
        $ne: null
      },
      monsters: {
        $ne: null
      }
    }
  },
  {
    $addFields: {
      total_oro: {
        $sum: "$loot.gold"
      },
      nivel_medio: {
        $avg: "$monsters.level"
      }
    }
  },
  {
    $addFields: {
      ratio_oro_nivel: {
        $divide: ["$total_oro", "$nivel_medio"]
      }
    }
  },
  {
    $sort: {
      ratio_oro_nivel: 1
    }
  },
  {
    $group: {
      _id: null,
      ratio_array: {
        $push: "$ratio_oro_nivel"
      },
      rooms: {
        $push: {
          _id: "$_id",
          ratio: "$ratio_oro_nivel"
        }
      }
    }
  },
  {
    $addFields: {
      cuartiles: {
        $percentile: {
          input: "$ratio_array",
          p: [0.25, 0.75],
          method: "approximate"
        }
      }
    }
  },
  {
    $addFields: {
      q1: {
        $arrayElemAt: ["$cuartiles", 0]
      },
      q3: {
        $arrayElemAt: ["$cuartiles", 1]
      }
    }
  },
  {
    $addFields: {
      iqr: {
        $subtract: ["$q3", "$q1"]
      }
    }
  },
  {
    $addFields: {
      lower: {
        $subtract: [
          "$q1",
          {
            $multiply: [1.5, "$iqr"]
          }
        ]
      },
      upper: {
        $add: [
          "$q3",
          {
            $multiply: [1.5, "$iqr"]
          }
        ]
      }
    }
  },
  {
    $unwind: "$rooms"
  },
  {
    $match: {
      $expr: {
        $or: [
          {
            $lt: ["$rooms.ratio", "$lower"]
          },
          {
            $gt: ["$rooms.ratio", "$upper"]
          }
        ]
      }
    }
  },
  {
    $lookup: {
      from: "rooms",
      localField: "rooms._id",
      foreignField: "_id",
      as: "original"
    }
  },
  {
    $unwind: "$original"
  },
  {
    $replaceRoot: {
      newRoot: "$original"
    }
  }
]